In [43]:
import pandas as pd
from pyspark.sql import SparkSession
import re
import os
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import current_date, to_timestamp, col, to_date, rpad, when, substring, lit
from notebookutils import mssparkutils
from pyspark.sql.functions import *

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 45, Finished, Available, Finished)

## Spark Logger enable for Notebook

In [44]:
logger = spark._jvm.org.apache.log4j.LogManager.getLogger("FabricLogger")

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 46, Finished, Available, Finished)

## Invoke Parameters From pipeline

In [45]:
# bronze layer file location
bronze_lakehouse_id = ''   # ce6ccccf-30b9-4340-acd0-d32d2ecc593a
bronze_workspace_id = ''  # 519f8648-7081-40b9-8281-fff5f7e0b555

# silver layer details
silver_workspace_id = ''  # 519f8648-7081-40b9-8281-fff5f7e0b555
silver_lakehouse_id = ''  # e215e7b4-5015-4798-8707-08971932ad2b
silver_lakehouse_name = ''# lhs_Product

# source file details
source_file = ''          # sales_orders20251220.csv
folder_path = ''          # Files/Product/Sales/202512

# table details
transformed_table_name = ''  # Sales_Orders_EXC
source_table_name = ''       # Sales_Orders_EXC_STG
table_schema = 'dbo'            # dbo

# schema file details
schema_path = ''             # Files/Schema/
schema_file_name = ''        # sales_order_schema.csv


StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 47, Finished, Available, Finished)

##  create file path for bronze file

In [46]:
# create full path to csv
source_filepath = ("abfss://"+ bronze_workspace_id+ "@onelake.dfs.fabric.microsoft.com/"+ bronze_lakehouse_id+ "/"+ folder_path+ "/")
source_fullpath = source_filepath + source_file

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 48, Finished, Available, Finished)

# Extract schema from schema file

In [47]:
schema_filepath =  "abfss://" + bronze_workspace_id + "@onelake.dfs.fabric.microsoft.com/" + bronze_lakehouse_id + "/" + schema_path+ "/"


try:
    df_ref_schema = spark.read.csv( schema_filepath + schema_file_name, header=True)
    df_ref_schema = df_ref_schema.withColumn( "data_type",lower(col("data_type")))
    logger.info("Schema file extract successfully")
except Exception as e:
    logger.error(f"error loading schema file: {str(e)}")
    raise e

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 49, Finished, Available, Finished)

## Source file count check

In [48]:
count = df_ref_schema.count()

if count == 0:
    raise Exception("schema file empty")

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 50, Finished, Available, Finished)

## Generate metadata from schema file

In [49]:
# Create PySpark schema
# Map string data types to PySpark types
type_map = {
    "string": StringType(),
    "char": StringType(),
    "num": IntegerType(),
    "int": IntegerType(),
    "double": DoubleType(),
    "boolean": BooleanType(),
    "long": LongType(),
    "float": FloatType(),
    "date": DateType(),
    "timestamp": StringType(),
    "Timestamp": StringType(),
    "datetime": StringType(),
    "DateTime": StringType(),
    "decimal": DoubleType()
}

# Create StructType schema
ref_schema = StructType([
    StructField(
        row.column_name,
        type_map.get(row.data_type.lower().strip(), StringType()),
        True
    )
    for row in df_ref_schema.collect()
])

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 51, Finished, Available, Finished)

## Read Data From Bronze Layer

In [50]:
try:
    df = spark.read.csv(source_fullpath, header=True, schema=ref_schema)
    logger.info("Bronze file extract successfully")
except Exception as e:
    logger.error(f"error loading bronze file: {str(e)}")
    raise

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 52, Finished, Available, Finished)

In [51]:
src_count = df.count()
logger.info(f"Source Row Count: {src_count}")
print(f"Source Row Count: {src_count}")

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 53, Finished, Available, Finished)

Source Row Count: 50


## Source File count check

In [52]:
count = df.count()
if count == 0:
    raise Exception("Source bronze file empty")

StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 54, Finished, Available, Finished)

## Load data to Silver Layer Parquet

In [53]:
try:
    silver_filename = source_file.replace(".csv", ".parquet")

    # write dataframe to parquet
    silver_path = "abfss://" + silver_workspace_id + "@onelake.dfs.fabric.microsoft.com/" + silver_lakehouse_id + "/Files/" + folder_path + "/" + silver_filename

    df.coalesce(1).write.mode("overwrite").parquet(silver_path)
    logger.info("Parquet file created in silver")
except Exception as e:
    logger.error(f"error loading parquet silver file: {str(e)}")
    raise e


StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 55, Finished, Available, Finished)

##  Load data to Silver Layer Transformed table

In [57]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

try:
    # Rename columns by stripping spaces and replacing inner spaces with underscores
    for col_name in df.columns:
        new_col_name = col_name.strip().replace(" ", "_")
        df = df.withColumnRenamed(col_name, new_col_name)

    # Clean and upper case some string columns (moved before window function)
    df = (
        df
        .withColumn("Region", f.upper(f.trim("Region")))
        .withColumn("SalesChannel", f.upper(f.trim("SalesChannel")))
        .withColumn("ProductCategory", f.upper(f.trim("ProductCategory")))
    )

    # Filter rows based on conditions
    df = df.filter(
        (f.col("OrderID").isNotNull()) &
        (f.col("Quantity") > 0) &
        (f.col("UnitPrice") > 0) &
        (f.col("OrderDate").isNotNull())
    )

    # Define window spec to get latest order per OrderID
    window_spec = Window.partitionBy("OrderID").orderBy(f.col("OrderDate").desc())

    df = (
        df
        .withColumn("rn", f.row_number().over(window_spec))
        .filter(f.col("rn") == 1)
        .drop("rn")
    )

    # Convert columns to appropriate data types
    df = (
        df
        .withColumn("OrderDate", f.to_date("OrderDate", "dd-MM-yyyy"))
        .withColumn("Quantity", f.col("Quantity").cast("int"))
        .withColumn("UnitPrice", f.col("UnitPrice").cast("double"))
    )

    # Calculate SalesAmount column
    df = df.withColumn(
        "SalesAmount",
        (f.col("Quantity") * f.col("UnitPrice")).cast("double")
    )

    # Add additional columns for analysis
    df = (
        df
        .withColumn("OrderYear", f.year("OrderDate"))
        .withColumn("OrderMonth", f.month("OrderDate"))
        .withColumn("Exdate", f.current_timestamp())
        .withColumn(
            "OrderValueBucket",
            f.when(f.col("SalesAmount") < 50000, "LOW")
             .when(f.col("SalesAmount") < 150000, "MEDIUM")
             .otherwise("HIGH")
        )
        .withColumn("Id", f.monotonically_increasing_id())
    )

    display(df)

    # Write dataframe to delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(
            f"abfss://{silver_workspace_id}@onelake.dfs.fabric.microsoft.com/"
            f"{silver_lakehouse_id}/Tables/{table_schema}/{transformed_table_name}"
        )

    logger.info("Silver layer data written successfully")

except Exception as e:
    logger.error(f"error loading transformed file: {str(e)}")
    raise


StatementMeta(, d9f7abe0-a1ff-41bf-a95b-6ea4742683d8, 59, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a817d1c7-69b3-4e61-a2ca-d03aa93b6842)